# Prototype 1: PDF → Markdown ingestion

Parses every PDF in `data/raw/` with MinerU and saves one Markdown file per PDF in `data/processed/markdown/`.

Flow: **config → list PDFs → parse + export → verify every page made it.**

Re-running is safe: a PDF is skipped only if its Markdown already contains *every* page.

In [ ]:
import re
import time
import json
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

from mineru import DoclibClient
from mineru.doclib import ParseRequest
from mineru.doclib.types import DocContentExportRequest

# ---- config ----
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
MARKDOWN_DIR = PROJECT_ROOT / "data" / "processed" / "markdown"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

TIER = "standard"      # one tier everywhere: flash | basic | standard | advanced
PAGE_RANGE = "all"     # IMPORTANT: MinerU parses only pages 1-10 if you don't say "all"
POLL_SECONDS = 5
TIMEOUT_SECONDS = 60 * 60   # per PDF; big PDFs can take a while

MARKDOWN_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

client = DoclibClient()

In [ ]:
pdf_files = sorted(RAW_DIR.glob("*.pdf"))
assert pdf_files, f"No PDFs found in {RAW_DIR}"

display(pd.DataFrame([
    {"filename": p.name, "size_mb": round(p.stat().st_size / 1024**2, 2)}
    for p in pdf_files
]))

## Helpers

- `pages_in_markdown` reads the `<!-- page X of N -->` markers MinerU writes, so we can tell if a file is complete.
- `parse_pdf` asks for **all** pages, waits (with a timeout), then exports the whole document in one call.

In [ ]:
PAGE_MARKER = re.compile(r"<!-- page (\d+) of (\d+) -->")


def pages_in_markdown(md_path):
    """Return (pages_found, total_pages) from MinerU page markers. (0, None) if unknown."""
    if not md_path.exists():
        return 0, None
    markers = PAGE_MARKER.findall(md_path.read_text(encoding="utf-8"))
    if not markers:
        return 0, None
    pages = {int(page) for page, _ in markers}
    total = int(markers[0][1])
    return len(pages), total


def is_complete(md_path):
    found, total = pages_in_markdown(md_path)
    return total is not None and found == total


def wait_for_parse(parse_id):
    deadline = time.time() + TIMEOUT_SECONDS
    parse = client.get_parse(parse_id)
    while parse.status in ("pending", "parsing"):
        if time.time() > deadline:
            raise TimeoutError(f"Parse {parse_id} still '{parse.status}' after {TIMEOUT_SECONDS}s")
        time.sleep(POLL_SECONDS)
        parse = client.get_parse(parse_id)
    if parse.status != "done":
        raise RuntimeError(f"Parse {parse_id} ended as {parse.status}: {parse.error_code} {parse.error_msg}")


def parse_pdf(pdf_path):
    md_path = MARKDOWN_DIR / f"{pdf_path.stem}.md"

    submit = client.ensure_parse(ParseRequest(
        path=str(pdf_path.resolve()),
        tier=TIER,
        page_range=PAGE_RANGE,
    ))
    for parse_id in submit.wait_parse_ids:
        wait_for_parse(parse_id)

    # Export the whole document straight to a .md file (no page-by-page loop needed)
    client.export_doc_content(
        submit.short_id,
        DocContentExportRequest(tier=TIER, page_range=PAGE_RANGE, output=str(md_path.resolve())),
    )

    found, total = pages_in_markdown(md_path)
    return {
        "document": pdf_path.name,
        "status": "success" if found == total else "incomplete",
        "pages": f"{found}/{total}",
        "characters": md_path.stat().st_size,
        "markdown_path": str(md_path),
    }

## Parse every PDF

Failures don't stop the loop; they're recorded in `artifacts/ingestion_manifest.json`.

In [ ]:
manifest = []

for pdf in tqdm(pdf_files, desc="Parsing PDFs"):
    md_path = MARKDOWN_DIR / f"{pdf.stem}.md"

    if is_complete(md_path):
        found, total = pages_in_markdown(md_path)
        manifest.append({"document": pdf.name, "status": "skipped_complete",
                         "pages": f"{found}/{total}", "markdown_path": str(md_path)})
        continue

    try:
        manifest.append(parse_pdf(pdf))
    except Exception as exc:
        manifest.append({"document": pdf.name, "status": "failed", "error": str(exc)})

    # save after every PDF so a crash doesn't lose progress
    (ARTIFACTS_DIR / "ingestion_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

display(pd.DataFrame(manifest))

## Verify: every raw PDF has a complete Markdown file

In [ ]:
report = []
for pdf in pdf_files:
    md_path = MARKDOWN_DIR / f"{pdf.stem}.md"
    found, total = pages_in_markdown(md_path)
    text = md_path.read_text(encoding="utf-8") if md_path.exists() else ""
    report.append({
        "pdf": pdf.name,
        "markdown_exists": md_path.exists(),
        "pages_found": found,
        "pages_total": total,
        "complete": total is not None and found == total,
        "words": len(text.split()),
        "headings": sum(line.lstrip().startswith("#") for line in text.splitlines()),
        "preview": " ".join(text.split())[:150],
    })

report_df = pd.DataFrame(report)
display(report_df)

missing = report_df.loc[~report_df["complete"], "pdf"].tolist()
assert not missing, f"Not fully parsed: {missing}. Re-run the parse cell (or check the manifest for errors)."
print(f"All {len(pdf_files)} PDFs fully parsed ✔")

# Part 2: Plain RAG baseline

```
markdown ─► clean ─► structure-aware sections ─► semantic chunks ─► chunks.jsonl
                                                                      │  (dedup by content hash)
                                                     Qdrant Cloud, embeddings computed server-side
                                                       dense  : all-MiniLM-L6-v2
                                                       sparse : qdrant/bm25
                                                       multi  : ColBERT (rerank only)
query ─► dense top-N + sparse top-N (prefetch) ─► ColBERT rerank ─► top-k, with section heading + page
```

**Setup:** put these two lines in a `.env` file in the project root. It's already git-ignored.
```
QDRANT_URL=https://xxxx.cloud.qdrant.io
QDRANT_API_KEY=your-key
```
Install: `pip install -U qdrant-client python-dotenv sentence-transformers`

In [ ]:
import os
import uuid
import hashlib
from collections import Counter
from itertools import groupby

import numpy as np
from dotenv import load_dotenv
from qdrant_client import QdrantClient, models

load_dotenv(PROJECT_ROOT / ".env")

# ---- paths ----
CLEAN_DIR = PROJECT_ROOT / "data" / "processed" / "clean"
CHUNKS_PATH = PROJECT_ROOT / "data" / "processed" / "chunks.jsonl"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# ---- semantic chunking (same knobs as Prototype4) ----
LOCAL_EMBED_MODEL = "all-MiniLM-L6-v2"   # only used locally to find topic shifts while chunking
SEMANTIC_PERCENTILE = 95                 # higher = fewer, bigger chunks
SEMANTIC_MIN_SENTENCES = 4               # shorter runs stay one chunk
MAX_CHUNK_CHARS = 1200                   # MiniLM reads ~256 tokens ≈ 1,000-1,200 chars, so the rest would be invisible
CHUNK_OVERLAP = 200
MIN_CHUNK_CHARS = 200                    # tiny chunks get merged into the previous one

# ---- Qdrant Cloud ----
QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
if not (QDRANT_URL and QDRANT_API_KEY):
    raise RuntimeError("QDRANT_URL / QDRANT_API_KEY missing. Put them in the .env file and restart the kernel.")

COLLECTION = "okf_chunks"
DENSE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
SPARSE_MODEL = "qdrant/bm25"
LATE_INTERACTION_MODEL = "answerdotai/answerai-colbert-small-v1"
PREFETCH_LIMIT = 30      # candidates from dense and from sparse
FINAL_TOP_K = 5
SKIP_UPLOAD = False      # True = don't touch Qdrant, just query

## Step 1: Cleaning

What the MinerU (flash) output contains that we don't want:
- image placeholders `![Formula block](doc:...)` and `<details>/<summary>image content</summary>` wrappers
- running page headers repeated on every page, e.g. *"Published in Transactions on Machine Learning Research"*
- page numbers and chart axis labels (`95`, `0.4`, `0.2`), which are lines that are only numbers
- footnote markers `<sup>11</sup>`
- HTML wrappers MinerU puts around page footnotes (`<small><span class="docvortex-page-footnote">`). The tags are stripped but the footnote **text is kept**, because some footnotes hold real content.
- table-of-contents lines with dot leaders (`. . . . . 141`), plus HTML entities like `&lt;` decoded to `<`

Page markers are kept for now because the next step needs them for page numbers.
Cleaned files are saved to `data/processed/clean/` so you can open and inspect them.

In [ ]:
import html

NOISE_LINE_RE = re.compile(
    r"^("
    r"!\[.*\]\(.*\)"                 # image / formula placeholders
    r"|</?details>"
    r"|<summary>.*</summary>"
    r"|[\d\s.,%()\-–]+"               # only numbers: page numbers, chart axes
    r"|.*(\. ){5,}.*"                   # table-of-contents dot leaders ". . . . . 141"
    r")$"
)
SUP_RE = re.compile(r"<sup>.*?</sup>")                                # footnote markers
HTML_TAG_RE = re.compile(r"</?(small|span|a|br|sub|i|b|em|u)\b[^>]*>")  # MinerU footnote wrappers etc.


def find_running_headers(lines, n_pages, min_share=0.3):
    """Lines repeated on >=30% of pages are page headers/footers, not content."""
    candidates = (
        l.strip() for l in lines
        if 15 < len(l.strip()) < 120 and not l.strip().startswith(("|", "```", "<!--", "#"))
    )
    counts = Counter(candidates)
    return {line for line, n in counts.items() if n >= max(5, min_share * n_pages)}


def clean_markdown(text):
    lines = text.splitlines()
    _, n_pages = PAGE_MARKER.findall(text)[0] if PAGE_MARKER.search(text) else (0, 1)
    headers = find_running_headers(lines, int(n_pages))

    kept, removed = [], 0
    for line in lines:
        s = line.strip()
        if s.startswith("<!-- Source page"):            # old marker from the first notebook version
            removed += 1
            continue
        if s and not s.startswith("<!--") and (NOISE_LINE_RE.match(s) or s in headers):
            removed += 1
            continue
        # drop footnote markers, strip HTML tags but keep their text, decode &lt; &amp; etc.
        kept.append(html.unescape(HTML_TAG_RE.sub("", SUP_RE.sub("", line))))

    cleaned = re.sub(r"\n{3,}", "\n\n", "\n".join(kept))   # collapse big gaps
    return cleaned, {"lines_removed": removed, "running_headers": sorted(headers)}


cleaning_report = []
for md_path in sorted(MARKDOWN_DIR.glob("*.md")):
    cleaned, info = clean_markdown(md_path.read_text(encoding="utf-8"))
    (CLEAN_DIR / md_path.name).write_text(cleaned, encoding="utf-8")
    cleaning_report.append({"doc": md_path.stem, **info,
                            "chars_before": md_path.stat().st_size, "chars_after": len(cleaned)})

display(pd.DataFrame(cleaning_report))

## Step 2: Structure-aware sections

MinerU flattened every heading to `##`, so the `#` count is useless for hierarchy. The **section number** gives it instead:
`3` → depth 1, `3.1` → depth 2, `B.2.1` → depth 3.

Rules:
- **Numbered heading** (`3.1 Law`, `A.6 Codeforces rating`): a real section. Update the section path.
- **Named heading** (Abstract, Introduction, Conclusion, Appendix…): a top-level section.
- **Any other `##`**: it's not a heading. MinerU marked dialogue lines like `## GPT-4:` or `## Juliet: ...` as headings, so these become normal text.
- **Contents / References**: skipped. A reference list is thousands of author names that pollute BM25 and the graph.

Each block keeps `doc`, `section` and `page`, the same idea as your `extract_blocks`.

In [ ]:
# "3.1 Law", "B.2.1 MS MARCO", "A GPT-4 has common sense grounding"
NUMBERED_RE = re.compile(r"^(?P<num>(?:\d{1,2}|[A-H])(?:\.\d{1,2})*)\.?\s+(?P<title>[A-Z][^\n]{1,150})$")
NAMED_SECTIONS = {"abstract", "introduction", "conclusion", "conclusions", "discussion",
                  "acknowledgments", "acknowledgements", "appendix", "contents",
                  "references", "bibliography"}
SKIP_SECTIONS = {"contents", "references", "bibliography"}


def as_section_heading(text):
    """Return (number, label) if `text` is a real section heading, else None.
    number is e.g. "3.1" for numbered sections, None for named ones like "Abstract"."""
    text = re.sub(r"\s+", " ", text).strip()
    if text.lower() in NAMED_SECTIONS:
        return None, text.title()
    m = NUMBERED_RE.match(text)
    if m and "," not in text:            # commas = author lists / citations, not headings
        return m["num"], f"{m['num']} {m['title']}"
    return None


def update_path(path, number, label):
    """path is a list of (number, label). Keep only ancestors of the new section.
    Uses the numbers (not position), so a missing "3" heading can't leave "3.1" under "2"."""
    if number is None:
        return [(None, label)]
    parents = [(n, l) for n, l in path if n is not None and number.startswith(n + ".")]
    return parents + [(number, label)]


def extract_sections(cleaned_text, doc):
    """Split cleaned markdown into blocks (paragraph / table / code) with section path + page."""
    title, path, page = doc, [], None
    blocks, buf, in_code = [], [], False

    def in_skip_section():
        return bool(path) and path[0][1].split()[-1].lower() in SKIP_SECTIONS

    def flush():
        nonlocal buf
        text = "\n".join(buf).strip()
        buf = []
        if text and not in_skip_section():
            blocks.append({"doc": doc, "section": " > ".join(l for _, l in path) or "Front matter",
                           "page": page, "text": text})

    for line in cleaned_text.splitlines():
        s = line.strip()

        if s.startswith("```"):                       # keep code blocks in one piece
            in_code = not in_code
            buf.append(line)
            continue
        if in_code:
            buf.append(line)
            continue

        if m := PAGE_MARKER.match(s):
            page = int(m.group(1))
            continue

        is_md_heading = s.startswith("#")
        heading_text = s.lstrip("#").strip()

        # Plain-text "References" / "A Appendix title" lines also count (AGI.md has them unmarked),
        # but only short lines without a trailing ":", so sentences are never mistaken for headings.
        if is_md_heading:
            heading = as_section_heading(heading_text.rstrip(":"))
        elif len(s) < 80 and not s.endswith(":"):
            heading = as_section_heading(s)
            # plain lines may only *start* references or *end* a skipped section
            if heading and not (heading[1].lower() in SKIP_SECTIONS or in_skip_section()):
                heading = None
        else:
            heading = None

        # first heading before any section = paper title ("# A Survey..." would otherwise look like appendix A)
        if is_md_heading and title == doc and not path and (s.startswith("# ") or heading is None):
            flush()
            title = heading_text
            continue

        if heading:
            flush()
            path = update_path(path, *heading)
            continue

        if not s:
            flush()
        else:
            buf.append(heading_text if is_md_heading else line)   # demoted "##" lines become text

    flush()
    return title, blocks


doc_titles, doc_blocks = {}, {}
for clean_path in sorted(CLEAN_DIR.glob("*.md")):
    title, blocks = extract_sections(clean_path.read_text(encoding="utf-8"), clean_path.stem)
    doc_titles[clean_path.stem], doc_blocks[clean_path.stem] = title, blocks

display(pd.DataFrame([
    {"doc": d, "title": doc_titles[d][:60], "blocks": len(b),
     "sections": len({x["section"] for x in b}), "words": sum(len(x["text"].split()) for x in b)}
    for d, b in doc_blocks.items()
]))

# peek at the section tree of one doc
print(*list(dict.fromkeys(b["section"] for b in doc_blocks["Holistic_Eval"]))[:15], sep="\n")

## Step 3: Semantic chunking (Prototype4 style)

Per section:
1. **Prose** runs are split into sentences and each sentence is embedded locally with MiniLM.
   We cut wherever the distance between neighbouring sentences is above the `SEMANTIC_PERCENTILE`, i.e. a topic shift.
2. Anything still over `MAX_CHUNK_CHARS` goes to the **sliding-window fallback**, which ends on a sentence boundary when it can and overlaps by `CHUNK_OVERLAP`.
   The fallback is also used if sections are too short, or if the embedding model can't load.
3. **Tables and code** skip the sentence splitter, since it would mash rows into one line. They're split by rows with the header row repeated.
4. Tiny leftovers are merged into the previous chunk of the same top-level section.

Every chunk keeps `section_path` and its **own** page range. Those become the headings in Qdrant.

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    _embedding_model = SentenceTransformer(LOCAL_EMBED_MODEL)
    print("Loaded", LOCAL_EMBED_MODEL)
except Exception as exc:
    _embedding_model = None
    print(f"Embedding model unavailable ({exc}); using fixed-size chunking only.")

_SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9])")
_SENTENCE_END_RE = re.compile(r"[.!?][\"')\]]?\s")


def norm(text):
    return re.sub(r"\s+", " ", text).strip()


def split_into_sentences(text):
    return [s.strip() for s in _SENTENCE_SPLIT_RE.split(text.strip()) if s.strip()]


def semantic_breakpoints(embeddings, percentile=SEMANTIC_PERCENTILE):
    a, b = embeddings[:-1], embeddings[1:]
    sims = (a * b).sum(axis=1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1) + 1e-8)
    distances = 1 - sims
    if len(distances) == 0:
        return []
    threshold = np.percentile(distances, percentile)
    return [i for i, d in enumerate(distances) if d > threshold]


def sliding_window(text, size=MAX_CHUNK_CHARS, overlap=CHUNK_OVERLAP):
    """Fixed-size fallback that prefers to end on a sentence boundary."""
    text = text.strip()
    if len(text) <= size:
        return [text] if text else []
    pieces, start = [], 0
    while start < len(text):
        end = min(start + size, len(text))
        if end < len(text):
            matches = list(_SENTENCE_END_RE.finditer(text[start:end]))
            if matches and matches[-1].end() > size // 2:
                end = start + matches[-1].end()
            else:
                space = text.rfind(" ", start, end)
                end = space if space > start else end
        pieces.append(text[start:end].strip())
        if end >= len(text):
            break
        next_start = max(end - overlap, start + 1)
        space = text.find(" ", next_start, end)
        start = space + 1 if space != -1 else next_start
    return [p for p in pieces if p]


def semantic_split(text):
    sentences = split_into_sentences(text)
    if _embedding_model is None or len(sentences) < SEMANTIC_MIN_SENTENCES:
        return sliding_window(text)
    emb = _embedding_model.encode(sentences, normalize_embeddings=True)
    pieces, start = [], 0
    for bp in semantic_breakpoints(emb) + [len(sentences) - 1]:
        piece = " ".join(sentences[start:bp + 1]).strip()
        start = bp + 1
        if piece:
            pieces += sliding_window(piece) if len(piece) > MAX_CHUNK_CHARS else [piece]
    return pieces


def is_table_or_code(text):
    return text.lstrip().startswith(("|", "```"))


def split_table(text):
    """Split a markdown table by rows, repeating the header so every piece is still a table."""
    if len(text) <= MAX_CHUNK_CHARS:
        return [text]
    lines = text.splitlines()
    if not lines[0].lstrip().startswith("|") or len(lines) < 3:      # code block
        return sliding_window(text)
    header, pieces, cur = lines[:2], [], []
    for row in lines[2:]:
        if cur and len("\n".join(header + cur + [row])) > MAX_CHUNK_CHARS:
            pieces.append("\n".join(header + cur))
            cur = []
        cur.append(row)
    if cur:
        pieces.append("\n".join(header + cur))
    return [p for piece in pieces for p in (sliding_window(piece) if len(piece) > MAX_CHUNK_CHARS else [piece])]


def chunk_prose(blocks):
    """Semantic-split a run of prose blocks; give each chunk its own page range."""
    full = " ".join(norm(b["text"]) for b in blocks)
    starts, pos = [], 0
    for b in blocks:
        starts.append((pos, b["page"]))
        pos += len(norm(b["text"])) + 1

    out, cursor = [], 0
    for piece in semantic_split(full):
        p = norm(piece)
        i = full.find(p[:60], cursor)
        i = cursor if i < 0 else i
        j = i + len(p)
        pages = [pg for s, pg in starts if s <= i][-1:] + [pg for s, pg in starts if i < s < j]
        out.append({"text": piece, "pages": [x for x in pages if x is not None]})
        cursor = i + 1
    return out


def chunk_document(blocks):
    raw = []
    for section, sec_blocks in groupby(blocks, key=lambda b: b["section"]):
        for is_table, run in groupby(list(sec_blocks), key=lambda b: is_table_or_code(b["text"])):
            run = list(run)
            if is_table:
                for b in run:
                    raw += [{"section": section, "text": t, "pages": [b["page"]] if b["page"] else []}
                            for t in split_table(b["text"])]
            else:
                raw += [{"section": section, **c} for c in chunk_prose(run)]

    merged = []
    for c in raw:
        prev = merged[-1] if merged else None
        same_top = prev and prev["section"].split(" > ")[0] == c["section"].split(" > ")[0]
        fits = prev and len(prev["text"]) + len(c["text"]) <= MAX_CHUNK_CHARS + MIN_CHUNK_CHARS
        if len(c["text"]) < MIN_CHUNK_CHARS and same_top and fits:
            prev["text"] += "\n\n" + c["text"]
            prev["pages"] += c["pages"]
        else:
            merged.append(dict(c))
    # drop leftovers with almost no words ("B.", lone symbols)
    return [c for c in merged if len(re.findall(r"[A-Za-z]{2,}", c["text"])) >= 5]


all_chunks = []
for doc, blocks in tqdm(doc_blocks.items(), desc="Chunking"):
    for i, c in enumerate(chunk_document(blocks)):
        pages = c["pages"]
        all_chunks.append({
            "chunk_id": f"{doc}_{i:04d}",
            "document": doc,
            "title": doc_titles[doc],
            "text": c["text"],
            "section_path": c["section"].split(" > "),
            "page_start": min(pages, default=None),
            "page_end": max(pages, default=None),
        })

with CHUNKS_PATH.open("w", encoding="utf-8") as f:
    for c in all_chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")
print(f"Saved {len(all_chunks)} chunks to {CHUNKS_PATH}")

### Check the outcome: chunking

In [ ]:
lengths = pd.DataFrame([{"document": c["document"], "chars": len(c["text"])} for c in all_chunks])
display(lengths.groupby("document")["chars"].describe()[["count", "mean", "min", "50%", "max"]].round(0))

oversized = [c for c in all_chunks if len(c["text"]) > MAX_CHUNK_CHARS + MIN_CHUNK_CHARS]
no_page = [c for c in all_chunks if c["page_start"] is None]
print(f"Total: {len(all_chunks)} | oversized: {len(oversized)} | without page: {len(no_page)}")

for c in pd.Series(all_chunks).sample(2, random_state=1):
    print(f"\n--- {c['chunk_id']} | {' > '.join(c['section_path'])} | p.{c['page_start']}-{c['page_end']}")
    print(c["text"][:400])

## Step 4: Qdrant Cloud: collection, dedup, upload

Same design as Prototype4:
- **3 named vectors**, all computed by Qdrant Cloud (`cloud_inference=True`): `dense`, `sparse` (BM25 with the IDF modifier), and `multi` (ColBERT, with HNSW disabled because it's only used for reranking).
- **Deterministic point IDs** (`uuid5(chunk_id)`) plus a **content hash**. Chunks are sorted into new, changed, unchanged or stale, and only new and changed ones get embedded. Stale points (chunks that no longer exist) are deleted.
- **`build_payload()`** is the one place a payload is defined. It carries `section_heading` and `page_label`, so retrieval can cite them.

In [ ]:
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, cloud_inference=True, timeout=120)

if not client.collection_exists(COLLECTION):
    client.create_collection(
        COLLECTION,
        vectors_config={
            "dense": models.VectorParams(size=384, distance=models.Distance.COSINE),
            "multi": models.VectorParams(
                size=96,
                distance=models.Distance.COSINE,
                multivector_config=models.MultiVectorConfig(comparator=models.MultiVectorComparator.MAX_SIM),
                hnsw_config=models.HnswConfigDiff(m=0),     # rerank only, never searched directly
            ),
        },
        sparse_vectors_config={"sparse": models.SparseVectorParams(modifier=models.Modifier.IDF)},
    )
    print(f"Created collection '{COLLECTION}'")
else:
    print(f"Collection '{COLLECTION}' exists")


def point_id(chunk_id):
    return str(uuid.uuid5(uuid.NAMESPACE_URL, chunk_id))


def content_hash(text):
    return hashlib.sha256(text.strip().encode("utf-8")).hexdigest()


def build_payload(chunk):
    start, end = chunk["page_start"], chunk["page_end"]
    if start is None:
        page_label = ""
    elif end is not None and end != start:
        page_label = f"pp.{start}–{end}"
    else:
        page_label = f"p.{start}"
    return {
        **chunk,
        "section_heading": " > ".join(chunk["section_path"]),
        "page_label": page_label,
        "content_hash": content_hash(chunk["text"]),
    }


def embed_text(chunk):
    """What gets embedded: the heading line + the chunk, so the vectors know the context."""
    return f"{chunk['title']} > {' > '.join(chunk['section_path'])}\n\n{chunk['text']}"


def stored_hashes():
    found, offset = {}, None
    while True:
        points, offset = client.scroll(COLLECTION, limit=256, offset=offset,
                                       with_payload=["content_hash"], with_vectors=False)
        found.update({str(p.id): (p.payload or {}).get("content_hash") for p in points})
        if offset is None:
            return found


stored = stored_hashes()
wanted = {point_id(c["chunk_id"]): c for c in all_chunks}

buckets = {"new": [], "changed": [], "unchanged": []}
for pid, c in wanted.items():
    if pid not in stored:
        buckets["new"].append(c)
    elif stored[pid] != content_hash(c["text"]):
        buckets["changed"].append(c)
    else:
        buckets["unchanged"].append(c)
stale = [pid for pid in stored if pid not in wanted]

print({k: len(v) for k, v in buckets.items()}, "| stale:", len(stale))
chunks_to_embed = buckets["new"] + buckets["changed"]

In [ ]:
if SKIP_UPLOAD:
    print("SKIP_UPLOAD=True, nothing sent to Qdrant")
else:
    if stale:
        client.delete(COLLECTION, points_selector=models.PointIdsList(points=stale))
        print(f"Deleted {len(stale)} stale points")

    BATCH = 32
    for i in tqdm(range(0, len(chunks_to_embed), BATCH), desc="Uploading"):
        batch = chunks_to_embed[i:i + BATCH]
        points = [
            models.PointStruct(
                id=point_id(c["chunk_id"]),
                vector={
                    "dense": models.Document(text=embed_text(c), model=DENSE_MODEL),
                    "sparse": models.Document(text=embed_text(c), model=SPARSE_MODEL),
                    "multi": models.Document(text=embed_text(c), model=LATE_INTERACTION_MODEL),
                },
                payload=build_payload(c),
            )
            for c in batch
        ]
        for attempt in range(3):                    # retry flaky cloud calls
            try:
                client.upsert(COLLECTION, points=points, wait=True)
                break
            except Exception as exc:
                if attempt == 2:
                    raise
                print(f"batch {i} failed ({exc}), retrying...")
                time.sleep(5 * (attempt + 1))

    print("Points in Qdrant:", client.count(COLLECTION, exact=True).count, "| chunks:", len(all_chunks))

### Check the outcome: is the section metadata actually in Qdrant?

In [ ]:
probe, _ = client.scroll(COLLECTION, limit=3, with_payload=True)
for p in probe:
    pl = p.payload or {}
    print(f"{pl.get('document')}  {pl.get('chunk_id')}")
    print(f"   heading: {pl.get('section_heading') or '(none)'}")
    print(f"   page:    {pl.get('page_label') or '(none)'}\n")

## Step 5: Hybrid retrieval + ColBERT rerank

Dense and sparse each **prefetch** `PREFETCH_LIMIT` candidates. ColBERT then rescores only that pool, so it never scans the whole collection, and the top `FINAL_TOP_K` come back with their payload (heading and page).

In [ ]:
def retrieve(query, top_k=FINAL_TOP_K):
    results = client.query_points(
        COLLECTION,
        prefetch=[
            models.Prefetch(query=models.Document(text=query, model=DENSE_MODEL),
                            using="dense", limit=PREFETCH_LIMIT),
            models.Prefetch(query=models.Document(text=query, model=SPARSE_MODEL),
                            using="sparse", limit=PREFETCH_LIMIT),
        ],
        query=models.Document(text=query, model=LATE_INTERACTION_MODEL),
        using="multi",
        limit=top_k,
        with_payload=True,
    )
    return results.points

### Check the outcome: retrieval, with headings

In [ ]:
test_queries = [
    "How does HELM measure calibration of language models?",
    "What did the GPT-4 technical report say about predictable scaling?",
    "What is homogenization in foundation models and why is it risky?",
    "How did the Sparks of AGI paper test GPT-4 on theory of mind?",
]

for query in test_queries:
    print(f"\nQ: {query}")
    for p in retrieve(query):
        pl = p.payload or {}
        print(f"  {p.score:7.3f}  {pl.get('title')}")
        print(f"           {pl.get('section_heading') or 'Unknown section'} ({pl.get('page_label') or 'p.?'})  [{pl.get('chunk_id')}]")